# Clan winner classifier

This notebook trains the classifier used by the Association advisor.

The important design choice is that both training and advisor scoring import the same feature-building code from `src/common/features.py`. This prevents the advisor from accidentally using a different feature table than the model was trained on.

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from xgboost import XGBClassifier

from src.common.features import prepare_members, build_matchup_feature_table

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)

RANDOM_STATE = 42

DATA_DIR = PROJECT_ROOT / 'data' / 'train'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MATCHES_FILE = DATA_DIR / 'clan_matches_training.csv'
MEMBERS_FILE = DATA_DIR / 'member_stats_training.csv'
MODEL_FILE = ARTIFACT_DIR / 'clan_winner_classifier_artifact.joblib'

In [6]:
VALUE_COLS = [
    'days_active_last_7_days',
    'days_since_last_active',
    'training_count_last_28_days',
    'avg_stars_top_11_players',
    'avg_stars_top_3_players',
    'avg_training_bonus',
]

members = pd.read_csv(MEMBERS_FILE)
matches = pd.read_csv(MATCHES_FILE)
members = prepare_members(members, value_cols=VALUE_COLS)

print('members:', members.shape)
print('matches:', matches.shape)

members: (291456, 15)
matches: (24288, 5)


## Build matchup-level training features

The raw data is split across two tables:

- `member_stats_training.csv`: one row per manager, six managers per clan
- `clan_matches_training.csv`: one row per clan-vs-clan matchup

The model cannot train directly on manager rows, so I first convert each clan roster into clan-level summary features. For each clan, managers are sorted consistently and summarized using mean, min, max, standard deviation, top-2 average, and bottom-2 average. I also create multiplier-weighted versions of the same summaries, because a manager's activity, training, or squad quality can matter more when their clan multiplier is higher.

After building features for both clans in a matchup, I create directional matchup features:

`clan_1_feature - clan_2_feature`

This keeps the training target simple: `target_clan1_win = 1` when clan 1 wins, otherwise `0`.

In [7]:
matchup, feature_cols = build_matchup_feature_table(
    members_df=members,
    matches_df=matches,
    value_cols=VALUE_COLS,
)

X = matchup[feature_cols].copy()
y = matchup['target_clan1_win'].astype(int)

print('X:', X.shape)
print('positive rate:', round(float(y.mean()), 4))

X.head(1)

X: (24288, 72)
positive rate: 0.4993


,days_active_last_7_days_mean_diff,days_active_last_7_days_min_diff,days_active_last_7_days_max_diff,days_active_last_7_days_std_diff,days_active_last_7_days_top2_avg_diff,days_active_last_7_days_bottom2_avg_diff,days_active_last_7_days_x_multiplier_mean_diff,days_active_last_7_days_x_multiplier_min_diff,days_active_last_7_days_x_multiplier_max_diff,days_active_last_7_days_x_multiplier_std_diff,days_active_last_7_days_x_multiplier_top2_avg_diff,days_active_last_7_days_x_multiplier_bottom2_avg_diff,days_since_last_active_mean_diff,days_since_last_active_min_diff,days_since_last_active_max_diff,days_since_last_active_std_diff,days_since_last_active_top2_avg_diff,days_since_last_active_bottom2_avg_diff,days_since_last_active_x_multiplier_mean_diff,days_since_last_active_x_multiplier_min_diff,days_since_last_active_x_multiplier_max_diff,days_since_last_active_x_multiplier_std_diff,days_since_last_active_x_multiplier_top2_avg_diff,days_since_last_active_x_multiplier_bottom2_avg_diff,training_count_last_28_days_mean_diff,training_count_last_28_days_min_diff,training_count_last_28_days_max_diff,training_count_last_28_days_std_diff,training_count_last_28_days_top2_avg_diff,training_count_last_28_days_bottom2_avg_diff,training_count_last_28_days_x_multiplier_mean_diff,training_count_last_28_days_x_multiplier_min_diff,training_count_last_28_days_x_multiplier_max_diff,training_count_last_28_days_x_multiplier_std_diff,training_count_last_28_days_x_multiplier_top2_avg_diff,training_count_last_28_days_x_multiplier_bottom2_avg_diff,avg_stars_top_11_players_mean_diff,avg_stars_top_11_players_min_diff,avg_stars_top_11_players_max_diff,avg_stars_top_11_players_std_diff,avg_stars_top_11_players_top2_avg_diff,avg_stars_top_11_players_bottom2_avg_diff,avg_stars_top_11_players_x_multiplier_mean_diff,avg_stars_top_11_players_x_multiplier_min_diff,avg_stars_top_11_players_x_multiplier_max_diff,avg_stars_top_11_players_x_multiplier_std_diff,avg_stars_top_11_players_x_multiplier_top2_avg_diff,avg_stars_top_11_players_x_multiplier_bottom2_avg_diff,avg_stars_top_3_players_mean_diff,avg_stars_top_3_players_min_diff,avg_stars_top_3_players_max_diff,avg_stars_top_3_players_std_diff,avg_stars_top_3_players_top2_avg_diff,avg_stars_top_3_players_bottom2_avg_diff,avg_stars_top_3_players_x_multiplier_mean_diff,avg_stars_top_3_players_x_multiplier_min_diff,avg_stars_top_3_players_x_multiplier_max_diff,avg_stars_top_3_players_x_multiplier_std_diff,avg_stars_top_3_players_x_multiplier_top2_avg_diff,avg_stars_top_3_players_x_multiplier_bottom2_avg_diff,avg_training_bonus_mean_diff,avg_training_bonus_min_diff,avg_training_bonus_max_diff,avg_training_bonus_std_diff,avg_training_bonus_top2_avg_diff,avg_training_bonus_bottom2_avg_diff,avg_training_bonus_x_multiplier_mean_diff,avg_training_bonus_x_multiplier_min_diff,avg_training_bonus_x_multiplier_max_diff,avg_training_bonus_x_multiplier_std_diff,avg_training_bonus_x_multiplier_top2_avg_diff,avg_training_bonus_x_multiplier_bottom2_avg_diff
0,0.166667,0.0,0.0,-0.505989,0.0,1.0,0.166667,0.0,0.0,-1.154655,0.0,2.0,-17.5,0.0,-42.0,-19.522089,0.0,-52.5,-7.0,0.0,-42.0,-15.652476,0.0,-21.0,-37.5,0.0,-88.0,-38.97002,-94.5,2.5,-113.333333,0.0,-264.0,-116.140572,-283.5,5.0,0.210897,0.898364,-0.031867,-0.458487,-0.381497,0.832727,1.28104,4.120436,-0.0956,-2.203222,-1.144491,4.44323,0.032489,0.5348,-0.717511,-0.50348,-0.774866,0.580445,1.153563,4.716845,-2.152532,-2.73786,-2.324599,4.909622,2.083333,0.0,0.5,-0.523231,0.5,4.5,4.75,0.0,1.5,-1.40723,1.5,9.0


In [8]:
base_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

param_grid = {
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.02, 0.03, 0.05],
    "n_estimators": [300, 600, 900],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

grid.fit(X, y)

best_model = grid.best_estimator_
print('best CV accuracy:', round(grid.best_score_, 6))
print('best params:', grid.best_params_)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits
best CV accuracy: 0.588974
best params: {'colsample_bytree': 0.8, 'learning_rate': 0.02, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 600, 'subsample': 1.0}


In [9]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 1)

oof_proba = np.zeros(len(X), dtype=float)
oof_pred = np.zeros(len(X), dtype=int)

for train_idx, valid_idx in outer_cv.split(X, y):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]

    fold_model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

    fold_grid = GridSearchCV(
        estimator=fold_model,
        param_grid=param_grid,
        scoring='accuracy',
        cv=inner_cv,
        n_jobs=-1,
        verbose=0,
        refit=True,
    )

    fold_grid.fit(X_train, y_train)

    valid_proba = fold_grid.best_estimator_.predict_proba(X_valid)[:, 1]
    valid_pred = (valid_proba >= 0.5).astype(int)

    oof_proba[valid_idx] = valid_proba
    oof_pred[valid_idx] = valid_pred

oof_accuracy = accuracy_score(y, oof_pred)
oof_logloss = log_loss(y, oof_proba)
oof_brier = brier_score_loss(y, oof_proba)

print('OOF accuracy:', round(oof_accuracy, 6))
print('OOF logloss :', round(oof_logloss, 6))

OOF accuracy: 0.586462
OOF logloss : 0.663437


## OOF accuracy by true match margin

The model performs differently depending on how decisive the match was. In the closest-margin bucket, accuracy is only slightly above random, while accuracy increases steadily as the absolute point difference becomes larger. This suggests that the available pre-match clan features contain useful signal for broader strength/preparation differences, but are much less informative for matches decided by small margins.

This pattern is consistent with the earlier EDA correlations: feature gaps are easier to detect in more decisive matches, while close matches are likely affected more by unobserved factors such as exact pairings, schedule choices, tactical decisions, and match-level variance.


In [10]:
eval_df = matchup[[
    "clan_1_id",
    "clan_2_id",
    "clan_1_points",
    "clan_2_points",
    "clan_winner",
    "target_clan1_win",
    "point_diff",
    "abs_point_diff",
]].copy()

eval_df["oof_proba_clan1_win"] = oof_proba
eval_df["oof_pred_clan1_win"] = oof_pred
eval_df["correct"] = (eval_df["oof_pred_clan1_win"] == eval_df["target_clan1_win"]).astype(int)

margin_bucket_labels = [
    "Q1_closest",
    "Q2",
    "Q3",
    "Q4",
    "Q5_largest",
]

eval_df["margin_bucket"] = pd.cut(
    eval_df["abs_point_diff"],
    bins=[0, 10, 20, 31, 47, np.inf],
    labels=margin_bucket_labels,
    include_lowest=True,
)

accuracy_by_margin_bucket = (
    eval_df
    .groupby("margin_bucket", observed=False)
    .agg(
        n_matches=("correct", "size"),
        share=("correct", lambda s: len(s) / len(eval_df)),
        accuracy=("correct", "mean"),
        mean_abs_point_diff=("abs_point_diff", "mean"),
        median_abs_point_diff=("abs_point_diff", "median"),
        min_abs_point_diff=("abs_point_diff", "min"),
        max_abs_point_diff=("abs_point_diff", "max"),
        mean_oof_proba=("oof_proba_clan1_win", "mean"),
    )
    .reset_index()
)

display(accuracy_by_margin_bucket)

,margin_bucket,n_matches,share,accuracy,mean_abs_point_diff,median_abs_point_diff,min_abs_point_diff,max_abs_point_diff,mean_oof_proba
0,Q1_closest,5117,0.210680,0.517100,5.882158,6.0,1,10,0.498491
1,Q2,4759,0.195940,0.541080,15.371296,15.0,11,20,0.499989
2,Q3,4799,0.197587,0.577620,25.734528,26.0,21,31,0.500165
3,Q4,4817,0.198328,0.619888,38.799460,39.0,32,47,0.501487
4,Q5_largest,4796,0.197464,0.680776,63.067556,60.0,48,144,0.497197


## Create test predictions

`GridSearchCV(refit=True)` has already refit the best model on the full training data.
I use `best_model = grid.best_estimator_` directly for the test submission.

In [11]:

TEST_DATA_DIR = PROJECT_ROOT / "data" / "test"
TEST_MATCHES_FILE = TEST_DATA_DIR / "clan_matches_test.csv"
TEST_MEMBERS_FILE = TEST_DATA_DIR / "member_stats_test.csv"

SUBMISSION_FILE = PROJECT_ROOT / "clan_winner_predictions.csv"

test_members = pd.read_csv(TEST_MEMBERS_FILE)
test_matches = pd.read_csv(TEST_MATCHES_FILE)

test_members = prepare_members(test_members, value_cols=VALUE_COLS)

test_matchup, test_feature_cols = build_matchup_feature_table(
    members_df=test_members,
    matches_df=test_matches,
    value_cols=VALUE_COLS,
)

X_test = test_matchup[feature_cols].copy()

test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred_clan1_win = (test_proba >= 0.5).astype(int)

submission = test_matchup[["clan_1_id", "clan_2_id"]].copy()
submission["predicted_clan_winner"] = np.where(
    test_pred_clan1_win == 1,
    1,
    2,
)

submission.to_csv(SUBMISSION_FILE, index=False)

print("test members:", test_members.shape)
print("test matches:", test_matches.shape)
print("X_test:", X_test.shape)


display(submission.head())

test members: (283752, 15)
test matches: (23646, 2)
X_test: (23646, 72)


,clan_1_id,clan_2_id,predicted_clan_winner
0,clan_5029188,clan_5045813,2
1,clan_5016270,clan_5010361,1
2,clan_5024341,clan_5015291,1
3,clan_5025374,clan_5002293,1
4,clan_5036635,clan_5029684,1


In [ ]:
artifact = {
    'model': best_model,
    'feature_cols': feature_cols,
    'value_cols': VALUE_COLS,
    'best_params': grid.best_params_,
}

joblib.dump(artifact, MODEL_FILE)